In [3]:
import pandas as pd

In [71]:
df=pd.read_csv(r"E:\Natural Language Processing\Dataset\Twitter_Data.csv")
df.head()

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [5]:
df.shape

(162980, 2)

In [6]:
df=df[:10000]

In [7]:
df.shape

(10000, 2)

In [8]:
df["category"].value_counts()

category
 1.0    4153
 0.0    3477
-1.0    2370
Name: count, dtype: int64

In [9]:
from nltk import word_tokenize
import string
from nltk import PorterStemmer
from nltk.corpus import stopwords

stemmer=PorterStemmer()


In [10]:
df["clean_text"]

0       when modi promised “minimum government maximum...
1       talk all the nonsense and continue all the dra...
2       what did just say vote for modi  welcome bjp t...
3       asking his supporters prefix chowkidar their n...
4       answer who among these the most powerful world...
                              ...                        
9995    modi made 1000 promises manifesto after electi...
9996                   jds leaders also saying modi modi 
9997    woh sirf modi gaali raha tha and changed his m...
9998    you must say what you witnessed since 2014 you...
9999    knows once modi comes again his entire family ...
Name: clean_text, Length: 10000, dtype: object

In [11]:
df["clean_text"]=df["clean_text"].astype("str")

In [12]:
def preprocess_text(text):
    text=text.lower()
    text=text.translate(str.maketrans('','',string.punctuation))
    doc=word_tokenize(text)
    stemmed=[stemmer.stem(i) for i in doc if i not in stopwords.words("english")]

    converted=" ".join(stemmed)
    return converted

In [13]:
df["clean_text"]=df["clean_text"].apply(preprocess_text)

In [14]:
df["clean_text"][:7]

0    modi promis “ minimum govern maximum govern ” ...
1                 talk nonsens continu drama vote modi
2    say vote modi welcom bjp told rahul main campa...
3    ask support prefix chowkidar name modi great s...
4    answer among power world leader today trump pu...
5              kiya tho refresh maarkefir comment karo
6    surat women perform yagna seek divin grace nar...
Name: clean_text, dtype: object

In [15]:
from gensim.utils import simple_preprocess  

X=df["clean_text"].apply(simple_preprocess)

In [16]:
import gensim

model=gensim.models.Word2Vec(
    window=10,
    workers=4,
    min_count=2
)
model.build_vocab(X,progress_per=1000)

In [17]:
model.train(X,total_examples=model.corpus_count,epochs=10)

(1192609, 1434640)

In [18]:
model.wv.most_similar("good")

[('well', 0.9694076180458069),
 ('doesnt', 0.9401350617408752),
 ('alway', 0.9303443431854248),
 ('know', 0.9291766285896301),
 ('noth', 0.9269552230834961),
 ('think', 0.9264591932296753),
 ('lot', 0.9244571328163147),
 ('realli', 0.9244512319564819),
 ('hope', 0.9205838441848755),
 ('els', 0.9145047068595886)]

In [19]:
model.wv.index_to_key

['modi',
 'india',
 'vote',
 'bjp',
 'peopl',
 'congress',
 'like',
 'elect',
 'year',
 'govt',
 'rahul',
 'narendra',
 'promis',
 'give',
 'money',
 'indian',
 'one',
 'get',
 'say',
 'dont',
 'want',
 'time',
 'poor',
 'countri',
 'make',
 'support',
 'lakh',
 'come',
 'know',
 'govern',
 'work',
 'nation',
 'parti',
 'chowkidar',
 'even',
 'gandhi',
 'also',
 'scheme',
 'need',
 'power',
 'everi',
 'ask',
 'via',
 'think',
 'see',
 'leader',
 'crore',
 'said',
 'per',
 'famili',
 'would',
 'minist',
 'good',
 'take',
 'back',
 'sir',
 'hai',
 'account',
 'call',
 'use',
 'polit',
 'right',
 'job',
 'becom',
 'never',
 'person',
 'better',
 'day',
 'pakistan',
 'done',
 'pleas',
 'farmer',
 'win',
 'much',
 'new',
 'last',
 'mani',
 'prime',
 'hindu',
 'still',
 'corrupt',
 'first',
 'question',
 'bank',
 'show',
 'made',
 'state',
 'thing',
 'announc',
 'let',
 'name',
 'muslim',
 'namo',
 'well',
 'fake',
 'campaign',
 'go',
 'bhakt',
 'media',
 'way',
 'world',
 'must',
 'may',
 '

In [20]:
X

0       [modi, promis, minimum, govern, maximum, gover...
1             [talk, nonsens, continu, drama, vote, modi]
2       [say, vote, modi, welcom, bjp, told, rahul, ma...
3       [ask, support, prefix, chowkidar, name, modi, ...
4       [answer, among, power, world, leader, today, t...
                              ...                        
9995    [modi, made, promis, manifesto, elect, manifes...
9996                  [jd, leader, also, say, modi, modi]
9997    [woh, sirf, modi, gaali, raha, tha, chang, min...
9998    [must, say, wit, sinc, still, see, poverti, ag...
9999    [know, modi, come, entir, famili, jail, corrup...
Name: clean_text, Length: 10000, dtype: object

In [21]:
import numpy as np

def get_avg_vec(tokens,model,size=100):
    vec=np.zeros((size))
    count=0
    for word in tokens:
        if word in model.wv.index_to_key:
            vec+=model.wv[word]
            count+= 1

    if(count):
        return vec/count
    else:
        return vec

In [22]:
trained=np.array([get_avg_vec(token_list,model,100) for token_list in X])

In [23]:
len(trained[0])

100

In [24]:
len(trained)

10000

In [25]:
y=df[["category"]][:10000]

In [26]:
type(y)

pandas.core.frame.DataFrame

In [27]:
y=np.ravel(y)

In [28]:
y

array([-1.,  0.,  1., ...,  1.,  1., -1.])

In [29]:
type(y)

numpy.ndarray

In [30]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(trained,y,test_size=.25)

In [31]:
x_train

array([[-0.29126806,  0.02441921, -0.07512627, ..., -0.29346656,
         0.01672638, -0.14671909],
       [-0.11762576,  0.14053725, -0.08744467, ..., -0.55705106,
        -0.00824777, -0.18364398],
       [ 0.16548908,  0.22654674, -0.11910818, ..., -0.71521685,
        -0.10292102, -0.23973203],
       ...,
       [ 0.03359196,  0.09162323, -0.02903687, ..., -0.5619094 ,
        -0.17983729, -0.36267261],
       [-0.4113002 ,  0.16352899,  0.01680401, ..., -0.19738116,
         0.21654106, -0.18393446],
       [-0.15492046,  0.05571593, -0.10756824, ..., -0.49696037,
         0.01341023, -0.19788108]])

In [32]:
from sklearn.ensemble import RandomForestClassifier

nlp=RandomForestClassifier(n_estimators=500)

nlp.fit(x_train,y_train)

RandomForestClassifier(n_estimators=500)

In [33]:
predict_y=nlp.predict(x_test)

In [34]:
from sklearn.metrics import  classification_report

print(classification_report(y_test,predict_y))

              precision    recall  f1-score   support

        -1.0       0.47      0.17      0.25       626
         0.0       0.60      0.57      0.59       847
         1.0       0.51      0.73      0.60      1027

    accuracy                           0.54      2500
   macro avg       0.53      0.49      0.48      2500
weighted avg       0.53      0.54      0.51      2500



In [43]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense,LSTM,Embedding
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer


tokenizer=Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(df["clean_text"])

In [45]:
word_index=tokenizer.word_index

In [46]:
word_index

{'<OOV>': 1,
 'modi': 2,
 'india': 3,
 '’': 4,
 'vote': 5,
 'bjp': 6,
 'peopl': 7,
 'congress': 8,
 'like': 9,
 'elect': 10,
 'govt': 11,
 'year': 12,
 'rahul': 13,
 'narendra': 14,
 'promis': 15,
 'give': 16,
 'money': 17,
 'indian': 18,
 'one': 19,
 'get': 20,
 'say': 21,
 'dont': 22,
 'want': 23,
 'time': 24,
 'poor': 25,
 'countri': 26,
 'make': 27,
 'support': 28,
 'come': 29,
 'know': 30,
 'lakh': 31,
 'govern': 32,
 'work': 33,
 'nation': 34,
 'parti': 35,
 'chowkidar': 36,
 'even': 37,
 'gandhi': 38,
 'also': 39,
 'scheme': 40,
 'need': 41,
 'power': 42,
 'everi': 43,
 'ask': 44,
 'via': 45,
 'think': 46,
 'see': 47,
 'leader': 48,
 'said': 49,
 'famili': 50,
 'would': 51,
 'minist': 52,
 'per': 53,
 'crore': 54,
 'good': 55,
 'take': 56,
 'back': 57,
 'sir': 58,
 'call': 59,
 'account': 60,
 'hai': 61,
 'use': 62,
 '2019': 63,
 'polit': 64,
 'job': 65,
 'right': 66,
 'becom': 67,
 'never': 68,
 'person': 69,
 'better': 70,
 'pakistan': 71,
 '2014': 72,
 'done': 73,
 'pleas': 7

In [47]:
df["clean_text"]

0       modi promis “ minimum govern maximum govern ” ...
1                    talk nonsens continu drama vote modi
2       say vote modi welcom bjp told rahul main campa...
3       ask support prefix chowkidar name modi great s...
4       answer among power world leader today trump pu...
                              ...                        
9995    modi made 1000 promis manifesto elect manifest...
9996                         jd leader also say modi modi
9997    woh sirf modi gaali raha tha chang mind gave c...
9998    must say wit sinc 2014 still see poverti agre ...
9999    know modi come entir famili jail corrupt way a...
Name: clean_text, Length: 10000, dtype: object

In [48]:
full_sequence=tokenizer.texts_to_sequences(df["clean_text"])

In [51]:
padded_full_sequence=pad_sequences(full_sequence,padding="pre",maxlen=100)

In [53]:
padded_full_sequence[1]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,  147,  967,  418, 1523,    5,
          2])

In [ ]:
vocab_size=len(word_index)+1
print(vocab_size)

15126


In [65]:
embedding_dim=20
max_length=100

dl_model=keras.Sequential(
    [
        Embedding(input_dim=vocab_size,output_dim=embedding_dim),

        LSTM(32),
        Dense(3,activation="softmax")
    ]
)
dl_model.build(input_shape=(None,max_length))
dl_model.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])

In [66]:
dl_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 100, 20)        │       302,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │         6,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 309,403 (1.18 MB)

 Trainable params: 309,403 (1.18 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
y=y+1

In [69]:
y

array([0., 1., 2., ..., 2., 2., 0.])

In [70]:
dl_model.fit(padded_full_sequence,y,epochs=10,batch_size=2)

Epoch 1/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 69s 14ms/step - accuracy: 0.5818 - loss: 0.8854
Epoch 2/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 82s 16ms/step - accuracy: 0.8634 - loss: 0.3845
Epoch 3/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 81s 16ms/step - accuracy: 0.9224 - loss: 0.2369
Epoch 4/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 83s 17ms/step - accuracy: 0.9506 - loss: 0.1628
Epoch 5/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 82s 16ms/step - accuracy: 0.9622 - loss: 0.1178
Epoch 6/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 82s 16ms/step - accuracy: 0.9733 - loss: 0.0776
Epoch 7/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 82s 16ms/step - accuracy: 0.9824 - loss: 0.0546
Epoch 8/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 81s 16ms/step - accuracy: 0.9906 - loss: 0.0322
Epoch 9/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 81s 16ms/step - accuracy: 0.9925 - loss: 0.0256
Epoch 10/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 83s 17ms/step - accuracy: 0.9935 - loss: 0.0189


In [72]:
test_x=df[10000:15000]

In [77]:
X_Test=test_x["clean_text"]

In [78]:
X_Test=tokenizer.texts_to_sequences(X_Test)

In [82]:
X_Test=pad_sequences(X_Test,padding="pre",maxlen=100)

In [94]:
y_Test=test_x["category"]
y_Test=y_Test+1


In [95]:
y_Test

10000    1.0
10001    1.0
10002    2.0
10003    2.0
10004    1.0
        ... 
14995    0.0
14996    0.0
14997    2.0
14998    1.0
14999    2.0
Name: category, Length: 5000, dtype: float64

In [85]:
predicted_y=dl_model.predict(X_Test)

157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [89]:
pred_classes=predicted_y.argmax(axis=1)

In [96]:
from sklearn.metrics import accuracy_score

print(accuracy_score(pred_classes,y_Test))

0.5816


In [90]:
final_pred=pred_classes-1

In [91]:
final_pred

array([ 0,  1,  1, ...,  1, -1, -1], dtype=int64)